In [ ]:
import os
import json
import openai
import numpy as np
import pandas as pd
import xes
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from langchain.vectorstores import Pinecone as PineconeVectorDB
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

In [ ]:
# Load XES event logs
def parse_xes(file_path):
    log = xes.parse(file_path)
    events = []
    for trace in log:
        for event in trace:
            events.append({
                "case_id": trace.attributes.get("concept:name", "Unknown"),
                "activity": event.get("concept:name", "Unknown"),
                "timestamp": event.get("time:timestamp", "Unknown"),
                "resource": event.get("org:resource", "Unknown")
            })
    return pd.DataFrame(events)

In [ ]:
# Convert event log to vector database
def create_vector_db(event_log_df, model_name="all-MiniLM-L6-v2", db_name="event_logs"):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(event_log_df.astype(str).agg(' '.join, axis=1).tolist())
    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
    index = pc.Index(db_name)
    for i, embedding in enumerate(embeddings):
        index.upsert(vectors=[(str(i), embedding.tolist(), event_log_df.iloc[i].to_dict())])
    return index


In [ ]:
# Retrieve relevant log entries
def retrieve_relevant(query, index, model_name="all-MiniLM-L6-v2", top_k=5):
    model = SentenceTransformer(model_name)
    query_embedding = model.encode(query).tolist()
    results = index.query(query_embedding, top_k=top_k, include_metadata=True)
    return [res["metadata"] for res in results]

In [ ]:
# Generate response using GPT-4
def generate_answer(query, retrieved_data):
    context = "\n".join(json.dumps(entry) for entry in retrieved_data)
    prompt = f"""
    Context:{context}
    Question: {query}
    
    Answer:
    """
    response = openai.ChatCompletion.create(
        model="gpt-4", messages=[{"role": "system", "content": "You are an expert in process mining."}, {"role": "user", "content": prompt}]
    )
    return response["choices"][0]["message"]["content"]

In [ ]:
# Example usage
if __name__ == "__main__":
    log_df = parse_xes("ContainerLogistics.xes")
    index = create_vector_db(log_df)
    query = "How many events are recorded in the log?"
    retrieved_data = retrieve_relevant(query, index)
    answer = generate_answer(query, retrieved_data)
    print(answer)